# Credit Risk Classification

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `default`

## 2 · Project Overview

We predict whether a loan applicant will **default** on their loan based on income, credit score, loan amount, employment history, existing debt, and home ownership.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a loan applicant's financial profile (income, credit score, loan amount, employment years, existing debt, home ownership), predict default risk.

## 5 · Why This Project Matters

- **Credit risk** assessment is the backbone of lending decisions.
- Defaults cost financial institutions billions annually.
- Regulatory requirements (Basel III) demand robust risk models.
- Understanding default drivers helps design better lending criteria.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | age, income, loan_amount, credit_score, employment_years, existing_debt, home_ownership |
| **Target** | default (0 = no default, 1 = default) |
| **Class balance** | ~80/20 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "default"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: default
Save dir: E:\Github\Machine-Learning-Projects\Classification\Credit Risk Classification


## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 loan applicants with financial and demographic features.

In [4]:
np.random.seed(SEED)
n = 8000
age = np.random.randint(21, 65, n)
income = np.round(np.random.lognormal(10.5, 0.5, n), 0).clip(15000, 200000)
loan_amount = np.round(np.random.lognormal(9.5, 0.7, n), 0).clip(1000, 100000)
credit_score = np.random.randint(300, 850, n)
employment_years = np.random.poisson(5, n).clip(0, 35)
existing_debt = np.round(np.random.exponential(5000, n).clip(0, 80000), 0)
home_ownership = np.random.choice(["Own", "Mortgage", "Rent"], n, p=[0.25, 0.4, 0.35])
own_num = np.where(home_ownership == "Own", 2, np.where(home_ownership == "Mortgage", 1, 0))

dti = existing_debt / (income + 1)
score = (-0.003 * credit_score + 0.00001 * loan_amount + 0.1 * dti
         - 0.02 * employment_years - 0.15 * own_num + np.random.normal(0, 0.5, n))
default = (score > np.percentile(score, 80)).astype(int)

df = pd.DataFrame({"age": age, "income": income, "loan_amount": loan_amount,
                    "credit_score": credit_score, "employment_years": employment_years,
                    "existing_debt": existing_debt, "home_ownership": home_ownership,
                    "default": default})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['default'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (8000, 8)
Class balance:
default
0    0.8
1    0.2
Name: proportion, dtype: float64


,age,income,loan_amount,credit_score,employment_years,existing_debt,home_ownership,default
0,59,31450.0,15509.0,650,2,4263.0,Mortgage,0
1,49,38760.0,8000.0,316,5,5771.0,Mortgage,1
2,35,41335.0,10656.0,514,4,4876.0,Rent,1
3,63,47276.0,23894.0,694,10,8315.0,Mortgage,0
4,28,15000.0,56060.0,698,3,11248.0,Mortgage,0


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (8000, 8)

Missing values:
age                 0
income              0
loan_amount         0
credit_score        0
employment_years    0
existing_debt       0
home_ownership      0
default             0
dtype: int64

Duplicate rows: 0

Dtypes:
age                   int32
income              float64
loan_amount         float64
credit_score          int32
employment_years      int32
existing_debt       float64
home_ownership       object
default               int64
dtype: object

Target 'default' confirmed. Value counts:
default
0    6400
1    1600
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["age", "income", "loan_amount", "credit_score", "employment_years", "existing_debt"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
plt.suptitle("Feature Distributions by Default Status", fontsize=14)
plt.tight_layout()
plt.show()

corr = df.select_dtypes(include="number").corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `default`.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind="bar", ax=axes[0], color=["steelblue", "salmon"], edgecolor="black")
axes[0].set_title("Default Distribution")
axes[0].set_xticklabels(["No Default (0)", "Default (1)"], rotation=0)

df.groupby("home_ownership")[TARGET].mean().plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Default Rate by Home Ownership")
plt.tight_layout()
plt.show()
print(f"Default rate: {df[TARGET].mean():.1%}")

Default rate: 20.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (6400, 7), Test: (1600, 7)
Train class distribution:
default
0    0.8
1    0.2
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `home_ownership` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Class imbalance**: ~20% default — use stratified split and consider class weights.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["dti_ratio"] = X_train["existing_debt"] / (X_train["income"] + 1)
X_test["dti_ratio"] = X_test["existing_debt"] / (X_test["income"] + 1)

X_train["loan_to_income"] = X_train["loan_amount"] / (X_train["income"] + 1)
X_test["loan_to_income"] = X_test["loan_amount"] / (X_test["income"] + 1)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (9): ['age', 'income', 'loan_amount', 'credit_score', 'employment_years', 'existing_debt', 'home_ownership', 'dti_ratio', 'loan_to_income']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.8213
F1       : 0.4479

              precision    recall  f1-score   support

           0       0.85      0.94      0.89      1280
           1       0.59      0.36      0.45       320

    accuracy                           0.82      1600
   macro avg       0.72      0.65      0.67      1600
weighted avg       0.80      0.82      0.80      1600



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                             Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                        
NearestCentroid              0.754375           0.784375  0.853308  0.776891   0.845275  0.754375    0.018278
CatBoostClassifier           0.829375           0.671875  0.860750  0.815993   0.812691  0.829375    3.048785
XGBClassifier                0.816875           0.665234  0.843110  0.806106   0.801048  0.816875    0.162050
LGBMClassifier               0.820000           0.661328  0.854944  0.807045   0.802368  0.820000    3.129378
RandomForestClassifier       0.821875           0.660156  0.852968  0.807904   0.803644  0.821875    1.109421
ExtraTreesClassifier         0.822500           0.658203  0.855841  0.807737   0.803762  0.822500    0.501203
LogisticRegression           0.821250           0.656250  0.861580  0.806383   0.80223

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.8287
F1: 0.4830


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.4601  (1.4s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[75]	valid_0's binary_logloss: 0.350022
LightGBM F1: 0.4604  (0.9s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
FLAML                  0.8287  0.4830     0.6095  0.4000
LightGBM               0.8213  0.4604     0.5810  0.3812
CatBoost               0.8225  0.4601     0.5874  0.3781
Logistic Regression    0.8213  0.4479     0.5859  0.3625

Top 3 models: ['FLAML', 'LightGBM', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  FLAML:
    Accuracy : 0.8287
    F1       : 0.4830
    Precision: 0.6095
    Recall   : 0.4000

  LightGBM:
    Accuracy : 0.8213
    F1       : 0.4604
    Precision: 0.5810
    Recall   : 0.3812

  CatBoost:
    Accuracy : 0.8225
    F1       : 0.4601
    Precision: 0.5874
    Recall   : 0.3781


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: FLAML

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.94      0.90      1280
           1       0.61      0.40      0.48       320

    accuracy                           0.83      1600
   macro avg       0.74      0.67      0.69      1600
weighted avg       0.81      0.83      0.81      1600


Total errors: 274 / 1600 (17.1%)

Sample misclassifications:
      age   income  loan_amount  credit_score  employment_years  existing_debt  home_ownership  dti_ratio  loan_to_income  actual  predicted  correct
3313   29  32654.0      24415.0           364                 4         8807.0             2.0   0.269698        0.747665       0          1    False
3574   37  19945.0       7521.0           345                 1        15937.0             2.0   0.799007        0.377068       0          1    False
1038   32  39142.0       6610.0           333                 1          163.0             0.0   0.004164        0.1688

## 25 · Interpretation and Business Insight

**Key findings:**
- **Credit score** is the strongest default predictor (lower = higher risk).
- **Debt-to-income ratio** is highly predictive.
- **Home ownership** signals financial stability.
- **Employment years** provide additional risk signal.

**Business takeaway:** Use credit score and DTI ratio as primary risk factors, with home ownership as a secondary indicator.

## 26 · Limitations

1. Synthetic data with simplified financial relationships.
2. No payment history or bureau data.
3. Missing loan purpose and terms.
4. No macroeconomic indicators.
5. 20% default rate is higher than real-world base rates.

## 27 · How to Improve This Project

1. Use real loan performance data (Lending Club, FICO).
2. Add payment history and bureau features.
3. Include loan purpose, term, and interest rate.
4. Add macroeconomic indicators (unemployment, interest rates).
5. Calibrate risk scores for regulatory compliance.

## 28 · Production Considerations

- Output calibrated default probabilities for pricing.
- Ensure regulatory compliance (fair lending, explainability).
- Monitor model performance across demographic groups.
- Retrain quarterly with recent loan performance data.
- Maintain model documentation for regulators.

## 29 · Common Mistakes

1. Using accuracy on imbalanced data (misleading).
2. Not considering cost asymmetry (approving a defaulter vs. rejecting a good borrower).
3. Ignoring fairness across protected groups.
4. Not calibrating probabilities for risk pricing.
5. Using variables that proxy for protected characteristics.

## 30 · Mini Challenge / Exercises

1. Remove `credit_score` — how much does F1 drop?
2. Compare DTI ratio vs. raw debt/income separately.
3. Try cost-sensitive learning with 5:1 false negative cost.
4. Plot PR curve and select optimal threshold.
5. Check for fairness across age groups.

## 31 · Final Summary / Key Takeaways

1. **Credit score** is the dominant default predictor.
2. **DTI ratio** is a powerful engineered feature.
3. **Class imbalance** requires PR-AUC and recall focus.
4. **Regulatory requirements** demand explainable models.
5. **Fairness auditing** is non-negotiable in credit.
6. **Calibrated probabilities** enable risk-based pricing.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Credit Risk Classification\metrics.json
{
  "CatBoost": {
    "accuracy": 0.8225,
    "f1": 0.4601,
    "precision": 0.5874,
    "recall": 0.3781
  },
  "LightGBM": {
    "accuracy": 0.8213,
    "f1": 0.4604,
    "precision": 0.581,
    "recall": 0.3812
  },
  "Logistic Regression": {
    "accuracy": 0.8213,
    "f1": 0.4479,
    "precision": 0.5859,
    "recall": 0.3625
  },
  "FLAML": {
    "accuracy": 0.8287,
    "f1": 0.483,
    "precision": 0.6095,
    "recall": 0.4
  }
}
